# Stage 1 — Merging tables. Your master dataframe joins: Invoice → InvoiceLine → Track → Genre + MediaType → Album → Artist, then attach Customer and Employee. This gives you one row per invoice-line with full context.


**Data Ingestion & Table Merging**

Before any ML, we need to understand what data we have and how it connects. Let me explain the thinking first.
What we're doing and why
The Chinook database has 11 tables, but ML models need a single flat table — one row per observation (e.g., one row per customer) with columns as features. So the first job is:

Connect to the SQLite .db file using Python
Load each table into a pandas DataFrame
Understand the relationships (which table joins to which)
Merge them into master DataFrames we'll use for ML

The table relationship map
Artist → Album → Track → InvoiceLine → Invoice → Customer → Employee
                    ↓
               Genre, MediaType
                    
Playlist → PlaylistTrack → Track
The core business flow is:
Customer bought something → recorded in Invoice → each item in InvoiceLine → each item is a Track → which belongs to an Album by an Artist in a Genre
What we'll build in Stage 1
Two master DataFrames:

df_transactions — every purchase line with full context (customer + track + genre + invoice details). This is the raw event log.
df_customers — one row per customer with aggregated stats. This becomes our ML feature table later.

Why not just merge everything at once?
Because some tables (like PlaylistTrack) are not relevant for purchase prediction. We only merge what adds signal for our ML problem. Joining irrelevant tables adds noise and computational cost.

In [73]:
import pandas as pd
import sqlite3


In [74]:
# 1. Connect to the database file
db_path = r"C:\Users\SANDE\Downloads\testing\Chinook.db"  # Replace with your actual file path
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
# 2. Query both tables and views from sqlite_master
# 'u' and 'v' represent standard tables 
query = "SELECT name FROM sqlite_master WHERE type IN ('table');"
cursor.execute(query)
names = [row[0] for row in cursor.fetchall()]
names

['Album',
 'Artist',
 'Customer',
 'Employee',
 'Genre',
 'Invoice',
 'InvoiceLine',
 'MediaType',
 'Playlist',
 'PlaylistTrack',
 'Track']

In [75]:
# Another way to get the table names is:
# 1. Connect to the database file
db_path = r"C:\Users\SANDE\Downloads\testing\Chinook.db"  # Replace with your actual file path
conn = sqlite3.connect(db_path)
query = "SELECT name FROM sqlite_master WHERE type IN ('table');"
table_names = pd.read_sql_query(query, conn)
print(table_names)
# table_names['name'][2]   # to get the name of the 3rd table using column-based indexing
# table_names.iloc[2].item()   # to get the name of the 3rd table using location-based indexing

             name
0           Album
1          Artist
2        Customer
3        Employee
4           Genre
5         Invoice
6     InvoiceLine
7       MediaType
8        Playlist
9   PlaylistTrack
10          Track


In [76]:
# Cell 3: Load every table into its own DataFrame
df_album        = pd.read_sql("SELECT * FROM Album", conn)
df_artist       = pd.read_sql("SELECT * FROM Artist", conn)
df_customer     = pd.read_sql("SELECT * FROM Customer", conn)
df_employee     = pd.read_sql("SELECT * FROM Employee", conn)
df_genre        = pd.read_sql("SELECT * FROM Genre", conn)
df_invoice      = pd.read_sql("SELECT * FROM Invoice", conn)
df_invoiceline  = pd.read_sql("SELECT * FROM InvoiceLine", conn)
df_mediatype    = pd.read_sql("SELECT * FROM MediaType", conn)
df_playlist     = pd.read_sql("SELECT * FROM Playlist", conn)
df_playlisttrack= pd.read_sql("SELECT * FROM PlaylistTrack", conn)
df_track        = pd.read_sql("SELECT * FROM Track", conn)

print("Tables loaded. Shapes:")
for name, df in [
    ("Album", df_album), ("Artist", df_artist), ("Customer", df_customer),
    ("Employee", df_employee), ("Genre", df_genre), ("Invoice", df_invoice),
    ("InvoiceLine", df_invoiceline), ("MediaType", df_mediatype),
    ("Playlist", df_playlist), ("PlaylistTrack", df_playlisttrack),
    ("Track", df_track)
]:
    print(f"  {name:15s} → {df.shape[0]} rows × {df.shape[1]} cols")

Tables loaded. Shapes:
  Album           → 347 rows × 3 cols
  Artist          → 275 rows × 2 cols
  Customer        → 59 rows × 13 cols
  Employee        → 8 rows × 15 cols
  Genre           → 25 rows × 2 cols
  Invoice         → 412 rows × 9 cols
  InvoiceLine     → 2240 rows × 5 cols
  MediaType       → 5 rows × 2 cols
  Playlist        → 18 rows × 2 cols
  PlaylistTrack   → 8715 rows × 2 cols
  Track           → 3503 rows × 9 cols


In [77]:
# Cell 4: Understand the key tables before merging

# The spine of our data — every purchase line item
print("=== InvoiceLine (the transaction backbone) ===")
print(df_invoiceline.head(3))
print(df_invoiceline.dtypes)
print()

print("=== Invoice (customer + date + total) ===")
print(df_invoice.head(3))
print(df_invoice.dtypes)
print()

print("=== Customer ===")
print(df_customer.head(3))
print()

print("=== Track ===")
print(df_track.head(3))

=== InvoiceLine (the transaction backbone) ===
   InvoiceLineId  InvoiceId  TrackId  UnitPrice  Quantity
0              1          1        2       0.99         1
1              2          1        4       0.99         1
2              3          2        6       0.99         1
InvoiceLineId      int64
InvoiceId          int64
TrackId            int64
UnitPrice        float64
Quantity           int64
dtype: object

=== Invoice (customer + date + total) ===
   InvoiceId  CustomerId          InvoiceDate           BillingAddress  \
0          1           2  2009-01-01 00:00:00  Theodor-Heuss-Straße 34   
1          2           4  2009-01-02 00:00:00         Ullevålsveien 14   
2          3           8  2009-01-03 00:00:00          Grétrystraat 63   

  BillingCity BillingState BillingCountry BillingPostalCode  Total  
0   Stuttgart         None        Germany             70174   1.98  
1        Oslo         None         Norway              0171   3.96  
2    Brussels         None        B

In [78]:
# Cell 5 (corrected): Merge tables into master transactions DataFrame

df_transactions = (
    df_invoiceline
    # Step A: Invoice info — rename InvoiceDate here directly
    .merge(
        df_invoice[["InvoiceId", "CustomerId", "InvoiceDate",
                    "BillingCountry", "Total"]].rename(
            columns={"InvoiceDate": "PurchaseDate"}),
        on="InvoiceId", how="left"
    )

    # Step B: Customer info
    .merge(
        df_customer[["CustomerId", "FirstName", "LastName",
                     "Country", "SupportRepId"]],
        on="CustomerId", how="left"
    )

    # Step C: Track info — rename UnitPrice and Name BEFORE merging
    .merge(
        df_track[["TrackId", "Name", "AlbumId", "GenreId",
                  "MediaTypeId", "Milliseconds", "UnitPrice"]].rename(
            columns={"Name": "TrackName", "UnitPrice": "TrackUnitPrice"}),
        on="TrackId", how="left"
    )

    # Step D: Album info — rename Title BEFORE merging
    .merge(
        df_album[["AlbumId", "Title", "ArtistId"]].rename(
            columns={"Title": "AlbumTitle"}),
        on="AlbumId", how="left"
    )

    # Step E: Artist info — rename Name BEFORE merging
    .merge(
        df_artist[["ArtistId", "Name"]].rename(
            columns={"Name": "ArtistName"}),
        on="ArtistId", how="left"
    )

    # Step F: Genre info — rename Name BEFORE merging
    .merge(
        df_genre[["GenreId", "Name"]].rename(
            columns={"Name": "GenreName"}),
        on="GenreId", how="left"
    )

    # Step G: Media type info — rename Name BEFORE merging
    .merge(
        df_mediatype[["MediaTypeId", "Name"]].rename(
            columns={"Name": "MediaTypeName"}),
        on="MediaTypeId", how="left"
    )
)

print("✓ Master transactions DataFrame built")
print(f"Shape: {df_transactions.shape}")
print("\nAll columns:")
for i, col in enumerate(df_transactions.columns):
    print(f"  {i:2d}  {col}")

✓ Master transactions DataFrame built
Shape: (2240, 24)

All columns:
   0  InvoiceLineId
   1  InvoiceId
   2  TrackId
   3  UnitPrice
   4  Quantity
   5  CustomerId
   6  PurchaseDate
   7  BillingCountry
   8  Total
   9  FirstName
  10  LastName
  11  Country
  12  SupportRepId
  13  TrackName
  14  AlbumId
  15  GenreId
  16  MediaTypeId
  17  Milliseconds
  18  TrackUnitPrice
  19  AlbumTitle
  20  ArtistId
  21  ArtistName
  22  GenreName
  23  MediaTypeName


In [79]:
# Diagnostic cell — run this before Cell 6
print("All columns in df_transactions:")
for i, col in enumerate(df_transactions.columns):
    print(f"  {i:2d}  {col}")

All columns in df_transactions:
   0  InvoiceLineId
   1  InvoiceId
   2  TrackId
   3  UnitPrice
   4  Quantity
   5  CustomerId
   6  PurchaseDate
   7  BillingCountry
   8  Total
   9  FirstName
  10  LastName
  11  Country
  12  SupportRepId
  13  TrackName
  14  AlbumId
  15  GenreId
  16  MediaTypeId
  17  Milliseconds
  18  TrackUnitPrice
  19  AlbumTitle
  20  ArtistId
  21  ArtistName
  22  GenreName
  23  MediaTypeName


In [80]:
# Cell 6: Convert types and add derived columns

# Convert PurchaseDate to datetime — essential for time-based features later
df_transactions["PurchaseDate"] = pd.to_datetime(df_transactions["PurchaseDate"])

# Create LineRevenue = price × quantity (actual revenue per line item)
# We use UnitPrice from InvoiceLine (what customer actually paid)
# NOT TrackUnitPrice (the track's listed price — could differ)
df_transactions["LineRevenue"] = df_transactions["UnitPrice"] * df_transactions["Quantity"]

# Verify the result
print("✓ Columns after cleaning:")
for i, col in enumerate(df_transactions.columns):
    print(f"  {i:2d}  {col}")

print("\nSample rows (transposed for readability):")
print(df_transactions.head(2))

print("\nData types:")
print(df_transactions.dtypes)

✓ Columns after cleaning:
   0  InvoiceLineId
   1  InvoiceId
   2  TrackId
   3  UnitPrice
   4  Quantity
   5  CustomerId
   6  PurchaseDate
   7  BillingCountry
   8  Total
   9  FirstName
  10  LastName
  11  Country
  12  SupportRepId
  13  TrackName
  14  AlbumId
  15  GenreId
  16  MediaTypeId
  17  Milliseconds
  18  TrackUnitPrice
  19  AlbumTitle
  20  ArtistId
  21  ArtistName
  22  GenreName
  23  MediaTypeName
  24  LineRevenue

Sample rows (transposed for readability):
   InvoiceLineId  InvoiceId  TrackId  UnitPrice  Quantity  CustomerId  \
0              1          1        2       0.99         1           2   
1              2          1        4       0.99         1           2   

  PurchaseDate BillingCountry  Total FirstName  ... GenreId MediaTypeId  \
0   2009-01-01        Germany   1.98    Leonie  ...       1           2   
1   2009-01-01        Germany   1.98    Leonie  ...       1           2   

   Milliseconds TrackUnitPrice         AlbumTitle  ArtistId  Artis

In [81]:
print("\nSample rows (transposed for readability):")
print(df_transactions.head(2))

# print("\nData types:")
# print(df_transactions.dtypes)


Sample rows (transposed for readability):
   InvoiceLineId  InvoiceId  TrackId  UnitPrice  Quantity  CustomerId  \
0              1          1        2       0.99         1           2   
1              2          1        4       0.99         1           2   

  PurchaseDate BillingCountry  Total FirstName  ... GenreId MediaTypeId  \
0   2009-01-01        Germany   1.98    Leonie  ...       1           2   
1   2009-01-01        Germany   1.98    Leonie  ...       1           2   

   Milliseconds TrackUnitPrice         AlbumTitle  ArtistId  ArtistName  \
0        342562           0.99  Balls to the Wall         2      Accept   
1        252051           0.99  Restless and Wild         2      Accept   

   GenreName             MediaTypeName LineRevenue  
0       Rock  Protected AAC audio file        0.99  
1       Rock  Protected AAC audio file        0.99  

[2 rows x 25 columns]


In [82]:
# Cell 7: Aggregate to one row per customer

df_customers = df_transactions.groupby("CustomerId").agg(
    FirstName       = ("FirstName",     "first"),
    LastName        = ("LastName",      "first"),
    Country         = ("Country",       "first"),
    SupportRepId    = ("SupportRepId",  "first"),
    TotalSpent      = ("LineRevenue",   "sum"),      # ML target variable
    NumInvoices     = ("InvoiceId",     "nunique"),  # how many times they bought
    NumTracks       = ("TrackId",       "count"),    # total items bought
    AvgInvoiceValue = ("Total",         "mean"),     # average basket size
    FavoriteGenre   = ("GenreName",     lambda x: x.value_counts().index[0]),
    FirstPurchase   = ("PurchaseDate",  "min"),
    LastPurchase    = ("PurchaseDate",  "max"),
).reset_index()

# Recency: days since last purchase
latest_date = df_transactions["PurchaseDate"].max()
df_customers["RecencyDays"] = (
    latest_date - df_customers["LastPurchase"]
).dt.days

# Tenure: how long they have been a customer in days
df_customers["TenureDays"] = (
    df_customers["LastPurchase"] - df_customers["FirstPurchase"]
).dt.days

print("✓ Customer summary table built")
print(f"Shape: {df_customers.shape}")
print("\nFirst few rows:")
print(df_customers.head())
print("\nKey stats:")
print(df_customers[["TotalSpent", "NumInvoices", "RecencyDays", "TenureDays"]].describe().round(2))

✓ Customer summary table built
Shape: (59, 14)

First few rows:
   CustomerId  FirstName     LastName         Country  SupportRepId  \
0           1       Luís    Gonçalves          Brazil             3   
1           2     Leonie       Köhler         Germany             5   
2           3   François     Tremblay          Canada             3   
3           4      Bjørn       Hansen          Norway             4   
4           5  František  Wichterlová  Czech Republic             4   

   TotalSpent  NumInvoices  NumTracks  AvgInvoiceValue FavoriteGenre  \
0       39.62            7         38         8.911053          Rock   
1       37.62            7         38         8.805789          Rock   
2       39.62            7         38         8.911053         Metal   
3       39.62            7         38         9.542632          Rock   
4       40.62            7         38         9.911053          Rock   

  FirstPurchase LastPurchase  RecencyDays  TenureDays  
0    2010-03-11   20

In [83]:
# Cell 8: Final checks and save to disk

# Check for missing values in both DataFrames
print("=== Missing values in df_transactions ===")
missing_trans = df_transactions.isnull().sum()
print(missing_trans[missing_trans > 0] if missing_trans.any() else "  None")

print("\n=== Missing values in df_customers ===")
missing_cust = df_customers.isnull().sum()
print(missing_cust[missing_cust > 0] if missing_cust.any() else "  None")

# Sanity checks
print("\n=== Sanity checks ===")
print(f"  Unique customers in transactions : {df_transactions['CustomerId'].nunique()}")
print(f"  Rows in df_customers             : {len(df_customers)}")
print(f"  These should match               : {df_transactions['CustomerId'].nunique() == len(df_customers)}")
print(f"  Total revenue in transactions    : ${df_transactions['LineRevenue'].sum():,.2f}")
print(f"  Total revenue in customers       : ${df_customers['TotalSpent'].sum():,.2f}")
print(f"  These should match               : {round(df_transactions['LineRevenue'].sum(), 2) == round(df_customers['TotalSpent'].sum(), 2)}")

# Save to disk
import os
data_path = os.path.join("..", "data")
os.makedirs(data_path, exist_ok=True)

df_transactions.to_csv(os.path.join(data_path, "transactions.csv"), index=False)
df_customers.to_csv(os.path.join(data_path, "customers.csv"), index=False)

conn.close()

print("\n✓ Saved:")
print(f"  transactions.csv → {df_transactions.shape}")
print(f"  customers.csv    → {df_customers.shape}")
print("✓ Database connection closed")

=== Missing values in df_transactions ===
  None

=== Missing values in df_customers ===
  None

=== Sanity checks ===
  Unique customers in transactions : 59
  Rows in df_customers             : 59
  These should match               : True
  Total revenue in transactions    : $2,328.60
  Total revenue in customers       : $2,328.60
  These should match               : True

✓ Saved:
  transactions.csv → (2240, 25)
  customers.csv    → (59, 14)
✓ Database connection closed


In [84]:
print(os.getcwd())

c:\Users\SANDE\Downloads\testing\Models


**Importing and Logging Parameters and Metrics **

In [85]:
import mlflow
mlflow.set_tracking_uri("http://127.0.0.1:5000")  # if using MLflow server
mlflow.set_experiment("Chinook_End_to_End_Project")
with mlflow.start_run(run_name="Data_Ingestion"):
    mlflow.log_param("source_database_is", "chinook.db")
    mlflow.log_param("Shape_of_customers_table", df_customers.shape)
    mlflow.log_param("Shape_of_transactions_table", df_transactions.shape)
    mlflow.log_artifact(os.path.join(data_path, "customers.csv"))

2026/06/04 00:26:25 INFO mlflow.tracking.fluent: Experiment with name 'Chinook_End_to_End_Project' does not exist. Creating a new experiment.


🏃 View run Data_Ingestion at: http://127.0.0.1:5000/#/experiments/970861043057358249/runs/c06c205b78b042dcbc7679ec52b40304
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/970861043057358249


In [86]:
# 4. Clean up connection
conn.close()
mlflow.end_run()